# 📘 Notebook 3 — Data Pipeline  *(Live Coding)*

**MiniMind Step-by-Step Series** | Milestone 3 of 6

> **Live Coding Session** — Synthetic data creation and DataLoader setup are pre-filled.
> Your job is to implement the **dataset classes** in the `# TODO` sections.

**Learning Objectives:**
- Implement `PretrainDataset` with BOS/EOS wrapping and padding
- Implement `apply_chat_template` for the ChatML format
- Implement `SFTDataset` with label masking on assistant responses


In [ ]:
# === Recap: Setup + Model Architecture (from Notebooks 1-2) ===
!pip install tokenizers torch transformers --quiet

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import json
import os
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader

print(f'PyTorch {torch.__version__}  CUDA: {torch.cuda.is_available()}')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- Config ---
@dataclass
class MiniMindConfig:
    hidden_size: int = 768
    num_hidden_layers: int = 8
    num_attention_heads: int = 8
    num_key_value_heads: int = 4
    head_dim: int = 96
    vocab_size: int = 6400
    intermediate_size: int = 2048
    max_position_embeddings: int = 32768
    rms_norm_eps: float = 1e-6
    rope_theta: float = 1e6
    dropout: float = 0.0
    hidden_act: str = 'silu'
    flash_attn: bool = True
    bos_token_id: int = 1
    eos_token_id: int = 2

# --- RMSNorm ---
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def norm(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
    def forward(self, x):
        return (self.weight * self.norm(x.float())).type_as(x)

# --- RoPE ---
def precompute_freqs_cis(dim, end=32768, rope_base=1e6):
    freqs = 1.0 / (rope_base ** (torch.arange(0, dim, 2)[:(dim // 2)].float() / dim))
    t = torch.arange(end)
    freqs = torch.outer(t, freqs).float()
    freqs_cos = torch.cat([torch.cos(freqs), torch.cos(freqs)], dim=-1)
    freqs_sin = torch.cat([torch.sin(freqs), torch.sin(freqs)], dim=-1)
    return freqs_cos, freqs_sin

def apply_rotary_pos_emb(q, k, cos, sin, unsqueeze_dim=1):
    def rotate_half(x):
        return torch.cat((-x[..., x.shape[-1] // 2:], x[..., :x.shape[-1] // 2]), dim=-1)
    cos, sin = cos.unsqueeze(unsqueeze_dim), sin.unsqueeze(unsqueeze_dim)
    return ((q * cos) + (rotate_half(q) * sin)).to(q.dtype), ((k * cos) + (rotate_half(k) * sin)).to(k.dtype)

# --- Attention ---
def repeat_kv(x, n_rep):
    bs, slen, n_kv_heads, head_dim = x.shape
    if n_rep == 1: return x
    return x[:, :, :, None, :].expand(bs, slen, n_kv_heads, n_rep, head_dim).reshape(bs, slen, n_kv_heads * n_rep, head_dim)

class Attention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_local_heads = config.num_attention_heads
        self.n_local_kv_heads = config.num_key_value_heads
        self.n_rep = self.n_local_heads // self.n_local_kv_heads
        self.head_dim = config.head_dim
        self.q_proj = nn.Linear(config.hidden_size, config.num_attention_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(config.hidden_size, config.num_key_value_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(config.hidden_size, config.num_key_value_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(config.num_attention_heads * self.head_dim, config.hidden_size, bias=False)
        self.q_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)
        self.k_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.dropout = config.dropout
        self.flash = hasattr(F, 'scaled_dot_product_attention') and config.flash_attn

    def forward(self, x, position_embeddings, past_key_value=None, use_cache=False, attention_mask=None):
        bsz, seq_len, _ = x.shape
        xq, xk, xv = self.q_proj(x), self.k_proj(x), self.v_proj(x)
        xq = xq.view(bsz, seq_len, self.n_local_heads, self.head_dim)
        xk = xk.view(bsz, seq_len, self.n_local_kv_heads, self.head_dim)
        xv = xv.view(bsz, seq_len, self.n_local_kv_heads, self.head_dim)
        xq, xk = self.q_norm(xq), self.k_norm(xk)
        cos, sin = position_embeddings
        xq, xk = apply_rotary_pos_emb(xq, xk, cos, sin)
        if past_key_value is not None:
            xk = torch.cat([past_key_value[0], xk], dim=1)
            xv = torch.cat([past_key_value[1], xv], dim=1)
        past_kv = (xk, xv) if use_cache else None
        xq, xk, xv = xq.transpose(1, 2), repeat_kv(xk, self.n_rep).transpose(1, 2), repeat_kv(xv, self.n_rep).transpose(1, 2)
        if self.flash and seq_len > 1 and past_key_value is None:
            output = F.scaled_dot_product_attention(xq, xk, xv, dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        else:
            scores = (xq @ xk.transpose(-2, -1)) / math.sqrt(self.head_dim)
            scores[:, :, :, -seq_len:] += torch.full((seq_len, seq_len), float("-inf"), device=scores.device).triu(1)
            if attention_mask is not None:
                scores += (1.0 - attention_mask.unsqueeze(1).unsqueeze(2)) * -1e9
            output = self.attn_dropout(F.softmax(scores.float(), dim=-1).type_as(xq)) @ xv
        output = output.transpose(1, 2).reshape(bsz, seq_len, -1)
        return self.resid_dropout(self.o_proj(output)), past_kv

# --- FeedForward (SwiGLU) ---
class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

# --- Block ---
class MiniMindBlock(nn.Module):
    def __init__(self, layer_id, config):
        super().__init__()
        self.self_attn = Attention(config)
        self.input_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.post_attention_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.mlp = FeedForward(config)
    def forward(self, hidden_states, position_embeddings, past_key_value=None, use_cache=False, attention_mask=None):
        residual = hidden_states
        hidden_states, present_kv = self.self_attn(self.input_layernorm(hidden_states), position_embeddings, past_key_value, use_cache, attention_mask)
        hidden_states = hidden_states + residual
        hidden_states = hidden_states + self.mlp(self.post_attention_layernorm(hidden_states))
        return hidden_states, present_kv

# --- Full Model ---
class MiniMindModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        self.dropout = nn.Dropout(config.dropout)
        self.layers = nn.ModuleList([MiniMindBlock(l, config) for l in range(config.num_hidden_layers)])
        self.norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        freqs_cos, freqs_sin = precompute_freqs_cis(dim=config.head_dim, end=config.max_position_embeddings, rope_base=config.rope_theta)
        self.register_buffer("freqs_cos", freqs_cos, persistent=False)
        self.register_buffer("freqs_sin", freqs_sin, persistent=False)
    def forward(self, input_ids, attention_mask=None, past_key_values=None, use_cache=False):
        batch_size, seq_length = input_ids.shape
        past_key_values = past_key_values or [None] * len(self.layers)
        start_pos = past_key_values[0][0].shape[1] if past_key_values[0] is not None else 0
        hidden_states = self.dropout(self.embed_tokens(input_ids))
        position_embeddings = (self.freqs_cos[start_pos:start_pos + seq_length], self.freqs_sin[start_pos:start_pos + seq_length])
        presents = []
        for layer, past_kv in zip(self.layers, past_key_values):
            hidden_states, present = layer(hidden_states, position_embeddings, past_key_value=past_kv, use_cache=use_cache, attention_mask=attention_mask)
            presents.append(present)
        return self.norm(hidden_states), presents

class MiniMindForCausalLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.model = MiniMindModel(config)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        self.model.embed_tokens.weight = self.lm_head.weight
    def forward(self, input_ids, attention_mask=None, labels=None, past_key_values=None, use_cache=False):
        hidden_states, past_key_values = self.model(input_ids, attention_mask, past_key_values, use_cache)
        logits = self.lm_head(hidden_states)
        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1), ignore_index=-100)
        return {'loss': loss, 'logits': logits, 'past_key_values': past_key_values}

config = MiniMindConfig()
print(f'✅ All prior code loaded — config: {config.hidden_size}d, {config.num_hidden_layers}L, {config.vocab_size}V')

## Part A — Creating Synthetic Training Data

Before building dataset classes, we need training data. In production, MiniMind trains on millions of text samples. For this notebook, we'll create small synthetic datasets to demonstrate the pipeline.

In [ ]:
# === Create Synthetic Pretraining Data ===
pretrain_texts = [
    "The transformer architecture was introduced in 2017.",
    "Language models learn to predict the next token in a sequence.",
    "Attention mechanisms allow models to focus on relevant parts of the input.",
    "Neural networks consist of layers of interconnected nodes.",
    "Gradient descent is the primary optimization algorithm for training neural networks.",
    "Backpropagation computes gradients by applying the chain rule.",
    "Tokenization converts raw text into numerical representations.",
    "The softmax function converts logits into probability distributions.",
    "Residual connections help train very deep networks.",
    "Layer normalization stabilizes training by normalizing activations.",
    "Dropout randomly zeroes activations to prevent overfitting.",
    "The learning rate controls the step size during optimization.",
    "Batch size determines how many samples are processed together.",
    "Cross-entropy loss measures the difference between predicted and true distributions.",
    "Weight decay adds L2 regularization to prevent overfitting.",
    "Mixed precision training uses float16 to speed up computation.",
    "The Adam optimizer combines momentum and adaptive learning rates.",
    "Cosine annealing gradually reduces the learning rate during training.",
    "Embedding layers map discrete tokens to continuous vectors.",
    "Position embeddings encode the order of tokens in a sequence.",
]

# Save as JSONL
os.makedirs('synthetic_data', exist_ok=True)
pretrain_path = 'synthetic_data/pretrain.jsonl'
with open(pretrain_path, 'w') as f:
    for text in pretrain_texts:
        f.write(json.dumps({'text': text}) + '\n')

print(f'✅ Created {len(pretrain_texts)} pretraining samples → {pretrain_path}')

In [ ]:
# === Create Synthetic SFT (Instruction) Data ===
sft_conversations = [
    [
        {"role": "user", "content": "What is a transformer?"},
        {"role": "assistant", "content": "A transformer is a neural network architecture that uses self-attention mechanisms to process sequential data in parallel, rather than sequentially like RNNs."}
    ],
    [
        {"role": "user", "content": "Explain attention in neural networks."},
        {"role": "assistant", "content": "Attention allows a model to dynamically focus on different parts of the input when producing each output element. It computes weighted sums of values based on query-key similarity scores."}
    ],
    [
        {"role": "user", "content": "What is tokenization?"},
        {"role": "assistant", "content": "Tokenization is the process of breaking text into smaller units called tokens. Common methods include BPE (Byte-Pair Encoding), which iteratively merges frequent character pairs."}
    ],
    [
        {"role": "user", "content": "How does gradient descent work?"},
        {"role": "assistant", "content": "Gradient descent updates model parameters by moving them in the direction that reduces the loss function. The step size is controlled by the learning rate."}
    ],
    [
        {"role": "user", "content": "What is overfitting?"},
        {"role": "assistant", "content": "Overfitting occurs when a model learns the training data too well, including noise and outliers, and fails to generalize to new unseen data. Techniques like dropout and weight decay help prevent it."}
    ],
    [
        {"role": "user", "content": "Explain the softmax function."},
        {"role": "assistant", "content": "Softmax converts a vector of real numbers into a probability distribution. Each output value is between 0 and 1, and all outputs sum to 1. It is commonly used in the final layer of classification models."}
    ],
    [
        {"role": "user", "content": "What is transfer learning?"},
        {"role": "assistant", "content": "Transfer learning involves taking a model trained on one task and adapting it to a different but related task. This saves computation and often improves performance, especially with limited data."}
    ],
    [
        {"role": "user", "content": "What is a loss function?"},
        {"role": "assistant", "content": "A loss function measures how far the model's predictions are from the true values. Common examples include cross-entropy for classification and mean squared error for regression."}
    ],
]

sft_path = 'synthetic_data/sft.jsonl'
with open(sft_path, 'w') as f:
    for conv in sft_conversations:
        f.write(json.dumps({'conversations': conv}) + '\n')

print(f'✅ Created {len(sft_conversations)} SFT samples → {sft_path}')

## Part B — PretrainDataset

The pretraining dataset is simple: each sample is a text passage wrapped with BOS and EOS tokens, padded to a fixed length. The model learns to predict each next token.

**Format:** `[BOS] token_1 token_2 ... token_n [EOS] [PAD] [PAD] ...`

**Labels:** Same as input_ids, but PAD positions are set to -100 (ignored by cross-entropy loss).

In [ ]:
# === Load Tokenizer ===
# For this notebook, we build a simple tokenizer using the HuggingFace
# tokenizers library — same approach as Notebook 1, but saved for reuse.
from tokenizers import decoders, models, pre_tokenizers, trainers, Tokenizer

# Build tokenizer (same as Notebook 1)
VOCAB_SIZE = 6400
all_special = [
    '<|endoftext|>', '<|im_start|>', '<|im_end|>', '<|im_sep|>',
    '<|endofprefix|>', '<|startoftext|>', '<|extra_0|>', '<|extra_1|>',
    '<|extra_2|>', '<|extra_3|>', '<|extra_4|>', '<|extra_5|>',
    '<|extra_6|>', '<|extra_7|>', '<|extra_8|>', '<|extra_9|>',
    '<pad>', '<mask>', '<sep>', '<cls>',
    '<unused_0>', '<unused_1>', '<unused_2>', '<unused_3>',
    '<unused_4>', '<unused_5>', '<unused_6>', '<unused_7>',
    '<think>', '</think>', '<tool_call>', '</tool_call>',
    '<tool_response>', '</tool_response>', '<code>', '</code>',
]

# Train on pretrain corpus for a functional tokenizer
with open(pretrain_path) as f:
    corpus = [json.loads(line)['text'] for line in f] * 100

raw_tokenizer = Tokenizer(models.BPE())
raw_tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
trainer_obj = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=all_special,
    show_progress=False,
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
)
raw_tokenizer.decoder = decoders.ByteLevel()
raw_tokenizer.train_from_iterator(corpus, trainer=trainer_obj)

# Create a wrapper with convenient attributes
class SimpleTokenizer:
    def __init__(self, tokenizer, special_tokens):
        self.tokenizer = tokenizer
        vocab = tokenizer.get_vocab()
        self.bos_token_id = vocab.get('<|im_start|>', 1)
        self.eos_token_id = vocab.get('<|im_end|>', 2)
        self.pad_token_id = vocab.get('<pad>', 0)

    def __call__(self, text, add_special_tokens=False, max_length=None, truncation=False):
        ids = self.tokenizer.encode(text).ids
        if max_length and truncation:
            ids = ids[:max_length]
        return type('Encoding', (), {'input_ids': ids})()

    def decode(self, ids):
        return self.tokenizer.decode(ids)

tokenizer = SimpleTokenizer(raw_tokenizer, all_special)
print(f'✅ Tokenizer ready — BOS={tokenizer.bos_token_id}, EOS={tokenizer.eos_token_id}, PAD={tokenizer.pad_token_id}')

# Quick test
test = tokenizer('Hello world')
print(f'   "Hello world" → {test.input_ids}')

In [ ]:
# === PretrainDataset ===
class PretrainDataset(Dataset):
    """Dataset for causal language model pretraining.

    Each sample: [BOS] + tokens + [EOS] + [PAD]*remaining
    Labels: same as input_ids, but PAD → -100 (ignored in loss)
    """
    def __init__(self, data_path, tokenizer, max_length=128):
        super().__init__()
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.samples = []
        with open(data_path) as f:
            for line in f:
                self.samples.append(json.loads(line))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]
        text = str(sample['text'])
        # ============================================================
        # TODO: Build input_ids and labels for one pretraining sample
        #
        # Steps:
        #   1. Tokenize (no special tokens, truncate to max_length - 2):
        #      tokens = self.tokenizer(text, add_special_tokens=False,
        #                              max_length=self.max_length - 2,
        #                              truncation=True).input_ids
        #   2. Wrap with BOS and EOS:
        #      tokens = [self.tokenizer.bos_token_id] + tokens + [self.tokenizer.eos_token_id]
        #   3. Pad to max_length using pad_token_id
        #   4. Convert to torch.tensor(..., dtype=torch.long) → input_ids
        #   5. Clone input_ids → labels; mask PAD positions with -100:
        #      labels[input_ids == self.tokenizer.pad_token_id] = -100
        #   6. Return (input_ids, labels)
        # ============================================================

        # YOUR CODE HERE
        pass

print('PretrainDataset defined \u2713')


In [ ]:
# === ✅ Milestone 3A: PretrainDataset ===
pretrain_ds = PretrainDataset(pretrain_path, tokenizer, max_length=64)

print(f'Dataset size: {len(pretrain_ds)}')
input_ids, labels = pretrain_ds[0]
print(f'Input shape:  {input_ids.shape}')
print(f'Labels shape: {labels.shape}')

# Verify structure
assert input_ids.shape == (64,), f'Expected shape (64,), got {input_ids.shape}'
assert labels.shape == (64,), f'Expected shape (64,), got {labels.shape}'
assert input_ids[0].item() == tokenizer.bos_token_id, 'First token should be BOS'

# Find EOS position
eos_positions = (input_ids == tokenizer.eos_token_id).nonzero()
assert len(eos_positions) > 0, 'Should have EOS token'

# Verify padding is masked in labels
pad_mask = (input_ids == tokenizer.pad_token_id)
if pad_mask.any():
    assert (labels[pad_mask] == -100).all(), 'PAD tokens should have label -100'

print(f'\n✅ Milestone 3A passed — PretrainDataset works')
print(f'   BOS at position 0, EOS at position {eos_positions[0].item()}')
print(f'   Content tokens: {eos_positions[0].item() - 1}')
print(f'   Padding tokens: {pad_mask.sum().item()}')

# Visualize a sample
print(f'\n--- Sample 0 ---')
print(f'Text: {pretrain_ds.samples[0]["text"]}')
print(f'IDs:  {input_ids[:20].tolist()}...')
print(f'Labels: {labels[:20].tolist()}...')

## Part C — SFTDataset (Supervised Fine-Tuning)

SFT data has a conversation format: user asks a question, assistant responds. The key insight is **label masking**: we only compute loss on the **assistant's response tokens**, not the user's question or system prompt.

**Why?** We want the model to learn *how to respond*, not memorize the questions.

**Format:**
```
<|im_start|>user\nWhat is X?<|im_end|>\n<|im_start|>assistant\nX is...<|im_end|>\n
```

**Labels:** -100 for everything except assistant response tokens.

In [ ]:
# === Chat Template for SFT ===
def apply_chat_template(messages):
    """Format messages into ChatML template."""
    # ============================================================
    # TODO: Build the ChatML-formatted string
    #
    # For each message (with 'role' and 'content'):
    #   Append: '<|im_start|>{role}\n{content}<|im_end|>\n'
    #
    # Return the full concatenated string.
    # ============================================================
    text = ''

    # YOUR CODE HERE

    return text

# Demo
demo_conv = [
    {"role": "user", "content": "What is AI?"},
    {"role": "assistant", "content": "AI is artificial intelligence."},
]
print(apply_chat_template(demo_conv))


In [ ]:
# === SFTDataset ===
class SFTDataset(Dataset):
    """Dataset for supervised fine-tuning with label masking.

    Only assistant responses contribute to the loss.
    User prompts and special tokens are masked with -100.
    """
    def __init__(self, jsonl_path, tokenizer, max_length=256):
        super().__init__()
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.samples = []
        with open(jsonl_path) as f:
            for line in f:
                self.samples.append(json.loads(line))

        # Marker token IDs for finding assistant response boundaries
        self.bos_marker = tokenizer('<|im_start|>assistant\n', add_special_tokens=False).input_ids
        self.eos_marker = tokenizer('<|im_end|>\n', add_special_tokens=False).input_ids

    def __len__(self):
        return len(self.samples)

    def generate_labels(self, input_ids):
        """Mask everything except assistant response regions with -100."""
        # ============================================================
        # TODO: Build the labels list for one sample
        #
        # Algorithm:
        #   Start with labels = [-100] * len(input_ids)
        #   Scan through input_ids with index i:
        #     - If input_ids[i : i+len(bos_marker)] == bos_marker:
        #         start = i + len(bos_marker)  (skip the marker itself)
        #         Scan forward from start to find eos_marker
        #         For positions start .. end+len(eos_marker):
        #             labels[j] = input_ids[j]  (unmask assistant tokens)
        #         Advance i past the eos_marker
        #     - Else: i += 1
        #   Return labels
        #
        # Key insight: only the assistant's response (and its EOS) is unmasked.
        # ============================================================
        labels = [-100] * len(input_ids)

        # YOUR CODE HERE

        return labels

    def __getitem__(self, index):
        sample = self.samples[index]
        conversations = sample['conversations']
        # ============================================================
        # TODO: Tokenize, pad, label-mask one conversation sample
        #
        # Steps:
        #   1. Format the conversation: prompt = apply_chat_template(conversations)
        #   2. Tokenize and truncate: input_ids = self.tokenizer(prompt).input_ids[:self.max_length]
        #   3. Pad to max_length with pad_token_id
        #   4. Generate masked labels: labels = self.generate_labels(input_ids)
        #   5. Return (torch.tensor(input_ids, dtype=torch.long),
        #              torch.tensor(labels, dtype=torch.long))
        # ============================================================

        # YOUR CODE HERE
        pass

print('SFTDataset defined \u2713')


In [ ]:
# === ✅ Milestone 3B: SFTDataset with Label Masking ===
sft_ds = SFTDataset(sft_path, tokenizer, max_length=128)

print(f'Dataset size: {len(sft_ds)}')
input_ids, labels = sft_ds[0]
print(f'Input shape:  {input_ids.shape}')
print(f'Labels shape: {labels.shape}')

assert input_ids.shape == (128,), f'Expected (128,), got {input_ids.shape}'
assert labels.shape == (128,), f'Expected (128,), got {labels.shape}'

# Count how many tokens have labels (assistant response tokens)
labeled_count = (labels != -100).sum().item()
total_non_pad = (input_ids != tokenizer.pad_token_id).sum().item()
print(f'\nTotal non-pad tokens: {total_non_pad}')
print(f'Labeled tokens (assistant): {labeled_count}')
print(f'Masked tokens (user/system): {total_non_pad - labeled_count}')

assert labeled_count > 0, 'Should have some labeled tokens'
assert labeled_count < total_non_pad, 'Not all tokens should be labeled'

print(f'\n✅ Milestone 3B passed — SFTDataset with label masking works')
print(f'   Only {labeled_count}/{total_non_pad} tokens contribute to loss')

# Visualize masking
print(f'\n--- Label Masking Visualization ---')
print(f'Conversation: {sft_ds.samples[0]["conversations"][0]["content"]}')
print(f'Response: {sft_ds.samples[0]["conversations"][1]["content"][:50]}...')
print(f'\nToken-by-token (first 40 positions):')
for i in range(min(40, total_non_pad)):
    tok_text = raw_tokenizer.decode([input_ids[i].item()])
    label_str = '✓ TRAIN' if labels[i].item() != -100 else '✗ mask '
    print(f'  {i:3d}: {label_str}  {tok_text!r}')

## Part D — DataLoader & Batching

PyTorch's DataLoader handles batching, shuffling, and parallel data loading. For training, we typically use:
- `batch_size`: number of samples per batch (e.g., 4-32)
- `shuffle=True`: randomize sample order each epoch
- `num_workers`: parallel data loading processes

In [ ]:
# === DataLoader ===
pretrain_loader = DataLoader(pretrain_ds, batch_size=4, shuffle=True)
sft_loader = DataLoader(sft_ds, batch_size=4, shuffle=True)

# Get a batch
pretrain_batch = next(iter(pretrain_loader))
sft_batch = next(iter(sft_loader))

print('=== Pretrain Batch ===')
print(f'  input_ids shape: {pretrain_batch[0].shape}')
print(f'  labels shape:    {pretrain_batch[1].shape}')

print(f'\n=== SFT Batch ===')
print(f'  input_ids shape: {sft_batch[0].shape}')
print(f'  labels shape:    {sft_batch[1].shape}')

# === ✅ Milestone 3C: DataLoader Verification ===
assert pretrain_batch[0].shape == (4, 64), f'Pretrain batch shape: {pretrain_batch[0].shape}'
assert sft_batch[0].shape == (4, 128), f'SFT batch shape: {sft_batch[0].shape}'

# Verify model can process the batch
config = MiniMindConfig()
model = MiniMindForCausalLM(config)
model.eval()

with torch.no_grad():
    outputs = model(pretrain_batch[0], labels=pretrain_batch[1])

assert outputs['loss'] is not None, 'Loss should not be None'
print(f'\n✅ Milestone 3C passed — DataLoader + model forward pass')
print(f'   Pretrain loss: {outputs["loss"].item():.4f}')
print(f'   Expected ~ln({config.vocab_size}) = {math.log(config.vocab_size):.2f} for random model')

## 🎯 Notebook 3 Summary

| Component | Key Concept |
|---|---|
| **Synthetic Data** | JSONL format for pretrain (text) and SFT (conversations) |
| **PretrainDataset** | [BOS] + tokens + [EOS] + [PAD], labels mask padding |
| **SFTDataset** | ChatML template, label masking on assistant responses only |
| **DataLoader** | Batching, shuffling, ready for training loop |

**Milestones completed:** 3A (PretrainDataset), 3B (SFTDataset with label masking), 3C (DataLoader + model)

### Next Notebook
In Notebook 4, we write the **training loop** — optimizer, learning rate scheduling, mixed-precision training, loss tracking, and checkpoint saving.